<a href="https://colab.research.google.com/github/sky-blue/manim/blob/main/manim_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!sudo apt install -y libcairo2-dev libpango1.0-dev ffmpeg texlive-latex-base texlive-fonts-recommended texlive-latex-extra
!pip install manim

Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:6 https://cli.github.com/packages stable InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
112 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not see

In [2]:

%%writefile scene.py
from manim import *


class FixedPoint(Scene):
    def construct(self):
        self.camera.background_color = "#101526"

        # 1. Title
        title = Text(
            "Fixed-Point",
            font_size=36,
            color=WHITE
        ).to_edge(UP).shift(DOWN * 0.25)

        self.play(Write(title))
        self.wait(0.5)

        # 2. Original binary representation
        decimal = MathTex(
            r"0.1_{10}",
            font_size=44
        )

        equal = MathTex(
            "=",
            font_size=44
        )

        binary_digits = "0.0001100110011"

        binary = MathTex(
            *list(binary_digits),
            font_size=44
        )

        original = VGroup(
            decimal,
            equal,
            binary
        ).arrange(RIGHT, buff=0.15)

        original.move_to(UP * 1.1)

        ellipsis = MathTex(
            r"\ldots",
            font_size=44
        ).next_to(binary, RIGHT, buff=0.08)

        self.play(
            Write(decimal),
            Write(equal),
            Write(binary),
            FadeIn(ellipsis),
            run_time=1.5
        )

        self.wait(1)

        # 3. Create 16-bit storage area
        cell_width = 0.43
        cell_height = 0.55
        cell_gap = 0.035

        def make_slots(count):
            return VGroup(*[
                Rectangle(
                    width=cell_width,
                    height=cell_height,
                    stroke_color=WHITE,
                    stroke_width=1.2
                )
                for _ in range(count)
            ]).arrange(RIGHT, buff=cell_gap)

        integer_slots = make_slots(8)
        fraction_slots = make_slots(8)

        bit_fields = VGroup(
            integer_slots,
            fraction_slots
        ).arrange(RIGHT, buff=0.35)

        bit_fields.move_to(DOWN * 0.7)

        integer_label = Text(
            "Integer Part",
            font_size=22,
            color=BLUE
        ).next_to(integer_slots, DOWN, buff=0.2)

        fraction_label = Text(
            "Fractional Part",
            font_size=22,
            color=GREEN
        ).next_to(fraction_slots, DOWN, buff=0.2)

        self.play(
            Create(bit_fields),
            FadeIn(integer_label),
            FadeIn(fraction_label),
            run_time=1.5
        )

        self.wait(1)

        # 4. Copy the integer part
        # The first 0 in 0.00011 is copied
        integer_bit = binary[0].copy()

        self.play(
            integer_bit.animate
            .set_color(BLUE)
            .move_to(integer_slots[0].get_center()),
            run_time=1.2
        )

        self.wait(0.5)

        # Fill the remaining integer slots with zeros
        integer_zeros = VGroup(*[
            integer_bit.copy()
            .set_color(BLUE)
            .move_to(integer_slots[i].get_center())
            for i in range(1, 8)
        ])

        self.play(
            LaggedStart(
                *[
                    FadeIn(bit)
                    for bit in integer_zeros
                ],
                lag_ratio=0.12
            ),
            run_time=1
        )

        self.wait(1)

        # 5. Copy the fractional part
        # 00011001 is copied into the fractional slots
        fraction_source = VGroup(*[
            binary[i].copy()
            for i in range(2, 10)
        ])

        fraction_targets = VGroup(*[
            bit.copy().set_color(GREEN)
            for bit in fraction_source
        ])

        for bit, slot in zip(fraction_targets, fraction_slots):
            bit.move_to(slot.get_center())

        self.play(
            LaggedStart(
                *[
                    Transform(source, target)
                    for source, target in zip(
                        fraction_source,
                        fraction_targets
                    )
                ],
                lag_ratio=0.08
            ),
            run_time=2
        )

        self.wait(2)


Overwriting scene.py


In [3]:
#!manim -pql scene.py FixedPoint
!manim -pqh scene.py FixedPoint

Manim Community v0.22.1

[11/21/26 05:13:12] INFO     Animation 0 : Partial      ]8;id=103029;file:///usr/local/lib/python3.13/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=113349;file:///usr/local/lib/python3.13/dist-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             movie file written in                              
                             '/content/media/videos/sce                         
                             ne/1080p60/partial_movie_f                         
                             iles/FixedPoint/f07208a261                         
                             f8471a03885dcb705f5fdbcb7c                         
                             5770e1be4aea257c6ed76b12f4                         
                             97.mp4'                                            
[11/21/26 05:13:14] INFO     Animation 1 : Partial      ]8;id=679803;file:///usr/local/lib/python3.13/dist-packages/ma

In [4]:
from IPython.display import Video, display

display(Video(
    "media/videos/scene/1080p60/FixedPoint.mp4",
    embed=True
))